# 🪶 Qareed (قريض) v1 — Arabic Poetry Fine-Tuning (QLoRA: LoRA adapters on a 4-bit base)

**Task:** Arabic instruction -> 1-4 baits of Arabic poetry. **Base:** `Qwen/Qwen2.5-1.5B-Instruct`. **Data:** `arbml/Ashaar_dataset`.

**Before you start**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Add a Colab Secret named `HF_ACCESS_TOKEN` (Hugging Face token with **Write** permission) and enable notebook access.
3. Run cells **top to bottom, one section at a time**. The training section takes about 1 hour on a T4.
4. Pushes to Hugging Face are **private** by default (`PUSH_PRIVATE = True` in section 0). Make a repo public only after you review the outputs.

**Known v1 limits (be honest in your model card):** meter is not controlled; theme labels are noisy; rhyme is weak; the source data lost the letter `إ` (patched for common words only).

## 0) Setup

In [9]:
# 0.1 install (restart the runtime if Colab asks, then continue from 0.2)
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes huggingface_hub pandas

In [10]:
# 0.2 GPU + versions
import torch, transformers, peft, trl, datasets
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__, "| datasets", datasets.__version__)

CUDA available: True
GPU: Tesla T4
VRAM (GB): 15.6
torch 2.11.0+cu130 | transformers 5.19.0 | peft 0.21.2 | trl 1.14.2 | datasets 5.1.0


In [11]:
# 0.3 config + Hugging Face login + Google Drive (checkpoints survive a Colab disconnect)
import os
from google.colab import userdata, drive
from huggingface_hub import login, whoami

BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
RUN_NAME     = "Qareed-Qwen2.5-1.5B-Poet-v1"
PUSH_PRIVATE = True        # keep True until you have reviewed the model

try:
    token = userdata.get("HF_ACCESS_TOKEN")
except Exception:
    token = userdata.get("HF_TOKEN")
login(token=token)
HF_USER = whoami()["name"]
print("Logged in as:", HF_USER)

drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/Colab Notebooks/Qareed"
os.makedirs(WORK_DIR, exist_ok=True)
OUT_DIR     = f"{WORK_DIR}/{RUN_NAME}"
ADAPTER_DIR = f"{OUT_DIR}/adapter"
ADAPTER_REPO = f"{HF_USER}/{RUN_NAME}-LoRA"
MERGED_REPO  = f"{HF_USER}/{RUN_NAME}"
print("Work dir:", WORK_DIR)

Logged in as: Youssefx64
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Work dir: /content/drive/MyDrive/Colab Notebooks/Qareed


## 1) Data: load + explore

Findings from exploration (kept here so you remember *why* the cleaning looks the way it does):
- `poem_verses` is a numpy array of **shatrs** (hemistichs). Two consecutive shatrs = one bait.
- `poem_meter` is a numeric id with no known name mapping -> **not used in v1**.
- `poem_theme` is missing for about 71% of poems; `قصيره` and `عامه` are not real themes.
- The source text has **no `إ` at all** (dropped upstream).

In [10]:
# 1.1 load dataset
import re, random, collections
import numpy as np
import pandas as pd
from datasets import load_dataset

raw = load_dataset("arbml/Ashaar_dataset", split="train")
df = raw.to_pandas()

COL = {"verses": "poem_verses", "theme": "poem_theme", "meter": "poem_meter",
       "era": "poet_era", "poet": "poet_name", "title": "poem_title"}

def shatrs_of(v):
    if isinstance(v, np.ndarray): v = v.tolist()
    if isinstance(v, (list, tuple)): return [str(x) for x in v]
    return str(v).split("\n")

print("Rows:", len(df), "| columns OK:", all(c in df.columns for c in COL.values()))

Rows: 212499 | columns OK: True


In [11]:
# 1.2 distributions (theme / era / language / shatr counts)
n_shatrs = df[COL["verses"]].map(lambda v: len(shatrs_of(v)))
print("Shatrs per poem:\n", n_shatrs.describe())
print("Odd-shatr poems:", (n_shatrs % 2 == 1).sum())
for c in ["poem_theme", "poet_era", "poem_language_type"]:
    print(f"\n=== {c} (missing: {df[c].isna().sum()}) ===")
    print(df[c].value_counts().head(15))

Shatrs per poem:
 count    212499.000000
mean         27.215497
std          66.006269
min           2.000000
25%           6.000000
50%          12.000000
75%          30.000000
max       11608.000000
Name: poem_verses, dtype: float64
Odd-shatr poems: 0

=== poem_theme (missing: 151608) ===
poem_theme
قصيدة قصيره      24602
قصيدة عامه       17324
قصيدة مدح         4835
قصيدة رومنسيه     3540
قصيدة حزينه       1966
قصيدة عتاب        1889
قصيدة هجاء        1538
قصيدة غزل         1289
قصيدة دينية       1014
قصيدة رثاء         837
قصيدة شوق          818
قصيدة فراق         552
قصيدة ذم           390
قصيدة وطنيه        268
قصيدة سياسية        12
Name: count, dtype: int64

=== poet_era (missing: 75799) ===
poet_era
العصر الحديث        48644
العصر العباسي       29935
العصر المملوكي      17690
العصر العثماني      10849
المغرب والأندلس      6461
العصر الفاطمي        4865
العصر الأندلسي       4524
العصر الأموي         3828
العصر الأيوبي        2986
المخضرمين            1881
العصر الجاهلي        

## 2) Cleaning (shatrs -> baits)

In [12]:
# 2.1 cleaning: pair shatrs, strip diacritics, patch missing hamza, drop noisy/split-word baits
STRIP_DIACRITICS = True
BAIT_JOINER = "  ...  "

TATWEEL = "\u0640"
DIAC = re.compile(r"[\u064B-\u0652\u0670\u0653-\u065F]")
ARABIC_RE = re.compile(r"[\u0600-\u06FF]")
LATIN_RE = re.compile(r"[A-Za-z]")
DIGIT_RE = re.compile(r"[0-9\u0660-\u0669]")

# words that only exist because the source dropped the letter 'إ'
FIX = {
    "لى": "إلى", "ن": "إن", "ون": "وإن", "نما": "إنما", "ني": "إني", "نني": "إنني",
    "نه": "إنه", "نك": "إنك", "نها": "إنها", "نهم": "إنهم", "نهما": "إنهما", "نكم": "إنكم",
    "مام": "إمام", "براهيم": "إبراهيم", "نسان": "إنسان", "يمان": "إيمان", "حسان": "إحسان",
    "ذ": "إذ",
}
SHORT_OK = {"و","من","في","عن","لا","ما","يا","لم","لن","إن","أن","قد","بل","أو","ثم","هل",
            "كي","لو","إذ","ذو","مع","أم","أي","به","له","بي","لي","هو","هي","كم","ها","فا","وا","ال"}

def fix_missing_hamza(s): return " ".join(FIX.get(w, w) for w in s.split())

def clean_shatr(s):
    s = s.replace(TATWEEL, "")
    if STRIP_DIACRITICS: s = DIAC.sub("", s)
    s = re.sub(r"\s+", " ", s).strip()
    return fix_missing_hamza(s)

def shatr_ok(s):
    if not (6 <= len(s) <= 80):  return False
    if LATIN_RE.search(s):       return False
    if DIGIT_RE.search(s):       return False
    return len(ARABIC_RE.findall(s)) / max(1, len(s.replace(" ", ""))) >= 0.9

bait_stats = collections.Counter()

def bait_reason(a, b):
    if not (shatr_ok(a) and shatr_ok(b)):             return "noisy_shatr"
    ta, tb = a.split(), b.split()
    if "ذا" in ta or "ذا" in tb:                      return "ambiguous_idha"   # idha/hadha lost its hamza: unsafe to guess
    if len(ta[-1]) <= 2 and ta[-1] not in SHORT_OK:   return "split_word"
    if len(tb[0]) <= 2 and tb[0] not in SHORT_OK:     return "split_word"
    return None

def poem_to_baits(v):
    shatrs = [clean_shatr(x) for x in shatrs_of(v)]
    if len(shatrs) % 2 == 1: shatrs = shatrs[:-1]
    baits = []
    for i in range(0, len(shatrs), 2):
        a, b = shatrs[i], shatrs[i + 1]
        r = bait_reason(a, b)
        bait_stats[r or "ok"] += 1
        baits.append(None if r else f"{a}{BAIT_JOINER}{b}")
    return baits

def norm_label(x, strip=("قصيدة ", "شعر ")):
    if x is None or (isinstance(x, float) and pd.isna(x)): return None
    x = re.sub(r"\s+", " ", str(x)).strip()
    for p in strip:
        if x.startswith(p): x = x[len(p):]
    return None if x.lower() in {"", "none", "nan", "other", "غير محدد"} else x

poems, dropped = [], collections.Counter()
for _, row in df.iterrows():
    baits = poem_to_baits(row[COL["verses"]])
    good = [b for b in baits if b]
    if len(good) < 2:                 dropped["too_few_good_baits"] += 1; continue
    if len(good) / len(baits) < 0.7:  dropped["noisy_poem"] += 1; continue
    poems.append({
        "baits": good,
        "theme": norm_label(row["poem_theme"]),
        "era":   norm_label(row["poet_era"]),
        "lang":  norm_label(row["poem_language_type"]),
        "meter": None,
    })
print("Bait stats:", dict(bait_stats))
print("Kept:", len(poems), "| Dropped:", dict(dropped))

seen, uniq = set(), []
for p in poems:
    key = " ".join(p["baits"][:2])
    if key not in seen: seen.add(key); uniq.append(p)
poems = uniq
print("After dedupe:", len(poems))

Bait stats: {'split_word': 160986, 'ok': 2575388, 'ambiguous_idha': 149687, 'noisy_shatr': 5572}
Kept: 178983 | Dropped: {'noisy_poem': 15502, 'too_few_good_baits': 18014}
After dedupe: 170927


## 3) Instruction dataset

In [13]:
# 3.1 prompt templates
import random
random.seed(42)

SYSTEM_PROMPT = (
    "أنت «قريض»، شاعر عربي متمكّن من اللغة والعروض والقافية. "
    "تكتب شعرًا عربيًا فصيحًا موزونًا ومقفّى، وتلتزم بعدد الأبيات المطلوب دون شرح أو تعليق."
)
NUM_WORDS = {1: "بيتًا واحدًا", 2: "بيتين", 3: "ثلاثة أبيات", 4: "أربعة أبيات"}

THEME_MAP = {
    "مدح":     ["المدح", "الثناء والمدح"],
    "رومنسيه": ["الرومانسية", "الحب والهيام"],
    "حزينه":   ["الحزن", "الحزن والأسى"],
    "عتاب":    ["العتاب", "العتاب واللوم"],
    "هجاء":    ["الهجاء", "الهجاء والتهكم"],
    "غزل":     ["الغزل", "الغزل بالمحبوبة"],
    "دينية":   ["الدين والتقوى", "الإيمان والدعاء"],
    "رثاء":    ["الرثاء", "رثاء الفقيد"],
    "شوق":     ["الشوق", "الشوق والحنين"],
    "فراق":    ["الفراق", "البعد والفراق"],
    "ذم":      ["الذم", "الذم واللوم"],
    "وطنيه":   ["حب الوطن", "الوطنية"],
    "سياسية":  ["السياسة"],
}
ERA_MAP = {"قبل الإسلام": "ما قبل الإسلام"}

T_THEME = [
    "اكتب {n} من الشعر في {theme}.",
    "قل لي {n} عن {theme}.",
    "أريد {n} من الشعر العربي في موضوع {theme}.",
    "انظم {n} في {theme}.",
    "هل تستطيع أن تكتب لي {n} عن {theme}؟",
]
T_THEME_ERA = [
    "اكتب {n} في {theme} بأسلوب شعراء {era}.",
    "انظم {n} في {theme} على طريقة شعراء {era}.",
]
T_ERA_ONLY = [
    "اكتب {n} من الشعر بأسلوب شعراء {era}.",
    "انظم {n} على طريقة شعراء {era}.",
]
T_FALLBACK = [
    "اكتب {n} من الشعر العربي.",
    "قل لي {n} من الشعر.",
]

def make_prompt(n, theme_key, era):
    n_txt = NUM_WORDS[n]
    theme = random.choice(THEME_MAP[theme_key]) if theme_key in THEME_MAP else None
    era_txt = ERA_MAP.get(era, era) if era else None
    options = []
    if theme:                 options += [(t, dict(theme=theme)) for t in T_THEME] * 3
    if theme and era_txt:     options += [(t, dict(theme=theme, era=era_txt)) for t in T_THEME_ERA]
    if era_txt and not theme: options += [(t, dict(era=era_txt)) for t in T_ERA_ONLY] * 2
    if not options:           options = [(t, {}) for t in T_FALLBACK]
    tpl, kw = random.choice(options)
    return tpl.format(n=n_txt, **kw)

for k in ["حزينه", "مدح", "هجاء"]:
    print(make_prompt(2, k, "العصر العباسي"))
print(make_prompt(2, None, "العصر المملوكي"))
print(make_prompt(2, None, None))

اكتب بيتين من الشعر في الحزن.
أريد بيتين من الشعر العربي في موضوع الثناء والمدح.
هل تستطيع أن تكتب لي بيتين عن الهجاء؟
اكتب بيتين من الشعر بأسلوب شعراء العصر المملوكي.
اكتب بيتين من الشعر العربي.


In [14]:
# 3.2 build train/val samples (language filter + keyword-verified theme windows, poem-level split)
BAD_LANG = {"عامي", "شعبي", "-"}
poems_f = [p for p in poems if p.get("lang") not in BAD_LANG]
print("poems after language filter:", len(poems_f))

# keyword lexicons per theme (my heuristics: review and edit them!)
THEME_KW = {
    "حزينه":   ["حزن","دمع","دموع","بكي","بكا","الأسى","أسى","ألم","لوعة","جوى","نحيب","أنين","شجن","وجع","يأس","كئيب"],
    "رثاء":    ["رثا","فقيد","المنون","منية","قبر","الردى","ردى","نعي","ضريح","ثكل","موت","مات","فقد","الحمام"],
    "غزل":     ["حبيب","لحظ","ثغر","جفن","جفون","هوى","عشق","غرام","وجنة","خدود","قوام","غزال","ريم","ظبي","فتنة","الحب","حبي"],
    "رومنسيه": ["حب","عشق","غرام","هوى","حبيب","شوق","لقاء"],
    "شوق":     ["شوق","حنين","اشتياق","أشتاق","وجد","لهف","جوى","لقاء","تذكر"],
    "فراق":    ["فراق","البين","وداع","رحيل","هجر","النوى","البعد","ودعت","غيابك","بعدك"],
    "دينية":   ["الله","ربي","دعاء","إله","النبي","محمد","رسول","صلاة","الجنة","ذنب","توبة","الإيمان","الدين","استغفر","رحمة","اللهم","سبحان"],
    "وطنيه":   ["وطن","الوطن","بلاد","أرض","شعب","الأمة","العرب","حرية","شهيد","مصر","فلسطين","القدس","عروبة","نضال","الجهاد"],
    "مدح":     ["كريم","جود","مجد","علا","شرف","أمير","فضل","همة","سماحة","الجود","الندى","سخاء","المعالي","الكرام","ثناء","عظيم"],
    "عتاب":    ["عتب","عتاب","لوم","جفوة","ظلم","خلان","صديق","وفاء","غدر","هجر","قسوة","لماذا","أخلف"],
    "هجاء":    ["لئيم","جبان","بخل","بخيل","خسيس","وضيع","ذليل","دنيء","نذل","حقير","جهل","كلب","حمار","خبيث","سفيه","فاجر"],
    "ذم":      ["لئيم","بخيل","سفيه","الدنيا","غدر","خيانة","خسيس","جاهل","نذل","ظالم","فاسد","خبث","لوم"],
}
THEME_MAP = {k: v for k, v in THEME_MAP.items() if k in THEME_KW}   # drops "سياسية" (only 12 poems)

def theme_ok(text, theme):
    return any(k in text for k in THEME_KW[theme])

MAX_PER_THEME = 2000
MAX_BAITS_THEMED = 12
P_START_AT_OPENING = 0.6
WINDOWS_THEMED = 4
TRIES_PER_WINDOW = 8
OTHER_WITH_ERA, OTHER_NO_ERA, WINDOWS_OTHER = 2500, 500, 2
N_CHOICES = [1, 2, 2, 2, 3, 4]
VAL_THEMED, VAL_OTHER = 600, 400
MAX_TRAIN_SAMPLES = 20000

def pick_window(baits, opening_bias):
    n = min(random.choice(N_CHOICES), len(baits))
    start = 0 if (opening_bias and random.random() < P_START_AT_OPENING) else random.randint(0, len(baits) - n)
    return n, baits[start:start + n]

def build_samples(poem, k, themed=False):
    out, theme = [], poem["theme"] if poem["theme"] in THEME_MAP else None
    for _ in range(k):
        for _try in range(TRIES_PER_WINDOW if (themed and theme) else 1):
            n, window = pick_window(poem["baits"], opening_bias=themed)
            if not (themed and theme) or theme_ok(" ".join(window), theme):
                out.append({"prompt": make_prompt(n, theme, poem["era"]),
                            "answer": "\n".join(window), "n_baits": n, "theme": theme})
                break
    return out

random.shuffle(poems_f)
themed = [p for p in poems_f if p["theme"] in THEME_MAP and len(p["baits"]) <= MAX_BAITS_THEMED]
other  = [p for p in poems_f if p["theme"] not in THEME_MAP]
print("themed poems:", len(themed), "| other poems:", len(other))

val_poems = themed[:VAL_THEMED] + other[:VAL_OTHER]          # poem-level split FIRST (no leakage)
themed_tr, other_tr = themed[VAL_THEMED:], other[VAL_OTHER:]

by_theme = collections.defaultdict(list)
for p in themed_tr: by_theme[p["theme"]].append(p)
themed_tr = [p for lst in by_theme.values() for p in lst[:MAX_PER_THEME]]

other_era   = [p for p in other_tr if p["era"]][:OTHER_WITH_ERA]
other_noera = [p for p in other_tr if not p["era"]][:OTHER_NO_ERA]

train_samples  = [s for p in themed_tr for s in build_samples(p, WINDOWS_THEMED, themed=True)]
train_samples += [s for p in other_era + other_noera for s in build_samples(p, WINDOWS_OTHER)]
random.shuffle(train_samples)
print("train samples (before cap):", len(train_samples))
train_samples = train_samples[:MAX_TRAIN_SAMPLES]

val_samples = [s for p in val_poems for s in build_samples(p, 1, themed=True)]
random.shuffle(val_samples)
print("train:", len(train_samples), "| val:", len(val_samples))
print(pd.Series([s["theme"] or "—none—" for s in train_samples]).value_counts())

for s in random.sample([x for x in train_samples if x["theme"]], 6):
    print("-" * 50); print("Q:", s["prompt"]); print("A:\n" + s["answer"])

poems after language filter: 164994
themed poems: 9522 | other poems: 148767
train samples (before cap): 21086
train: 20000 | val: 651
—none—     5678
رومنسيه    3282
مدح        3077
حزينه      1619
غزل        1195
دينية      1150
عتاب       1028
شوق         816
رثاء        761
فراق        630
هجاء        419
وطنيه       254
ذم           91
Name: count, dtype: int64
--------------------------------------------------
Q: انظم ثلاثة أبيات في الحزن والأسى.
A:
هزج الأصائل يستحث كؤوسها  ...  ويقيمنا للفرض في الأسحار
لهفا على القمري لهفا دائما  ...  يكوي الحشى بجوى كلذع النار
ولقد هجرت الصبر بعد فراقه  ...  ولقد مزجت دما بدمع جارى
--------------------------------------------------
Q: اكتب ثلاثة أبيات من الشعر في الإيمان والدعاء.
A:
رأيت في الدين راء موفقة  ...  فأنزل الله قرنا يزكيها
وكنت أول من قرت بصحبته  ...  عين الحنيفة واجتازت أمانيها
قد كنت أعدى أعاديها فصرت لها  ...  بنعمة الله حصنا من أعاديها
--------------------------------------------------
Q: هل تستطيع أن تكتب لي ثلاثة أبيات عن الف

In [15]:
# 3.3 tokenizer + chat template + length check
from transformers import AutoTokenizer
from datasets import Dataset

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def to_text(s):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": s["prompt"]},
        {"role": "assistant", "content": s["answer"]},
    ]
    return tok.apply_chat_template(messages, tokenize=False)

train_ds = Dataset.from_list([{"text": to_text(s)} for s in train_samples])
val_ds   = Dataset.from_list([{"text": to_text(s)} for s in val_samples])

lens = [len(tok(t["text"]).input_ids) for t in train_ds.select(range(2000))]
print("Token length — mean:", int(np.mean(lens)), "| p95:", int(np.percentile(lens, 95)), "| max:", max(lens))
print(train_ds[0]["text"])

Token length — mean: 153 | p95: 202 | max: 221
<|im_start|>system
أنت «قريض»، شاعر عربي متمكّن من اللغة والعروض والقافية. تكتب شعرًا عربيًا فصيحًا موزونًا ومقفّى، وتلتزم بعدد الأبيات المطلوب دون شرح أو تعليق.<|im_end|>
<|im_start|>user
انظم بيتين في العتاب.<|im_end|>
<|im_start|>assistant
عتبت على دهري بأفعاله التي  ...  براني بها بري السهام من الهم
ليصرف عني فادحات نوائب  ...  أضاق بها ذرعي وأضنى بها جسمي<|im_end|>



In [16]:
# 3.4 save the exact train/val data to Drive (reproducibility + safe resume)
import json
for name, data in [("train", train_samples), ("val", val_samples)]:
    with open(f"{WORK_DIR}/qareed_{name}.jsonl", "w", encoding="utf-8") as f:
        for s in data:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")
print("saved", len(train_samples), len(val_samples), "->", WORK_DIR)

saved 20000 651 -> /content/drive/MyDrive/Colab Notebooks/Qareed


## 4) Model + baseline (before training)

If you ran a training attempt earlier in this session, **restart the runtime** (or clean up) before loading the model again. Loading the model again on top of an old `trainer` causes adapters to stack.

In [17]:
# 4.1 cleanup (safe to run even on a fresh session)
import gc, torch
for name in ["trainer", "model"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
print("GPU mem (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

GPU mem (GB): 0.0


In [18]:
# 4.2 load model in 4-bit (QLoRA)
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4: fp16 is the fast path
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto", dtype=torch.float16,
)
model.config.use_cache = False
print("Loaded. Memory (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded. Memory (GB): 1.15


In [19]:
# 4.3 generation helper + test prompts + BASELINE
def generate(prompt, model, max_new_tokens=160, temperature=0.8, top_p=0.9, rep_pen=1.1):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=temperature, top_p=top_p, repetition_penalty=rep_pen,
                             pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

TEST_PROMPTS = [
    "اكتب بيتين من الشعر في الحزن.",
    "انظم ثلاثة أبيات في الفخر.",               # unseen theme
    "اكتب بيتًا واحدًا عن الصبر.",               # unseen theme
    "أريد بيتين من الشعر العربي في الغزل.",
    "اكتب بيتين في مدح العلم.",                  # unseen phrasing
    "اكتب بيتين من الشعر بأسلوب شعراء العصر العباسي.",
    "انظم بيتين في الشوق والحنين.",
    "اكتب أربعة أبيات في رثاء الأم.",            # unseen phrasing
]

print("===== BASELINE (before fine-tuning) =====")
for p in TEST_PROMPTS:
    print("\nQ:", p); print(generate(p, model))

===== BASELINE (before fine-tuning) =====

Q: اكتب بيتين من الشعر في الحزن.
في لحظة من اللحظات  
تختلط البسمة بالمأساة  
الحنان يُخَلْق في القلب  
ومكان الصمت عمياء الحقيقة

Q: انظم ثلاثة أبيات في الفخر.
في فخرها كثيف وفي نورها دايموند،
في عيونها لامات وذوبان في خيره.
في روحها بحر وغابة، لا يزال فيها الغم.
في خيرها سفر ومغيب، ولا تنسى الحب.

Q: اكتب بيتًا واحدًا عن الصبر.
"ومن صبر، يوفق في أمره ويحصد".

Q: أريد بيتين من الشعر العربي في الغزل.
أنا أتفهم أنك تبحث عن الشعور بالأدب وتألق النثر، لكن الابتعاد عن التعقيد اللغوي يمكن أن يحسن الإنتاج. سأساعدك بنظرة قصيرة على كيفية كتابة البيتان:

البيتان:

1. أنت نسيم الجبال،
2. لا تغيب الشمس حتى.

هذان البيتانان صغيران وممتنعين. الأول يربط بينك وبين الطبيعة، بينما الثاني يربط بينك وبين الزمن الذي تولد فيه. إذا كنت ترغب في ابتكار بيتان آخر، فستكون قادر على ذلك باستخدام نفس الأسلوب البسيط والدقيق. يمكنك استخدام العبارات التي تصف حالة الحب وال

Q: اكتب بيتين في مدح العلم.
في عزِل البحار، يُعلَمَ السباق، وفي غابات الصخر، يُعلَمَ القُرة.

في نور ال

## 5) Training (QLoRA)

- `FAST = False` (batch 8 + gradient checkpointing) is the config verified on a T4: ~2.2 GB peak, ~62 min for 1250 steps.
- `FAST = True` (batch 16, no checkpointing) **ran out of memory on a T4** (Qwen's large vocabulary makes the logits heavy). Keep it `False`.
- Do **not** run `Run all` through this section twice on the same `model`: reload the model first (section 4).

In [20]:
# 5.1 LoRA + trainer config
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

FAST = False   # keep False on a T4 (FAST=True caused CUDA OOM)

peft_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

speed = dict(per_device_train_batch_size=16, per_device_eval_batch_size=16,
             gradient_accumulation_steps=1, gradient_checkpointing=False) if FAST else \
        dict(per_device_train_batch_size=8, per_device_eval_batch_size=8,
             gradient_accumulation_steps=2, gradient_checkpointing=True,
             gradient_checkpointing_kwargs={"use_reentrant": False})

args = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=40,                 # newer transformers dropped warmup_ratio
    weight_decay=0.01,
    max_grad_norm=0.3,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=20,
    eval_strategy="steps", eval_steps=200,
    save_strategy="steps", save_steps=200, save_total_limit=2,
    report_to="none",
    dataset_text_field="text",
    max_length=256,
    packing=False,
    seed=42,
    **speed,
)

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     eval_dataset=val_ds, processing_class=tok, peft_config=peft_cfg)
trainer.model.print_trainable_parameters()

Adding EOS to train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/651 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/651 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/651 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/651 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [21]:
# 5.2 LoRA weights -> fp32 (the fp16 GradScaler cannot handle bf16 grads; REQUIRED on this setup)
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

c = collections.Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad)
print("trainable dtypes:", dict(c))      # expect only torch.float32

trainable dtypes: {'torch.float32': 392}


In [22]:
# 5.3 train (resumes from the last checkpoint on Drive if Colab disconnected)
import glob, time
ckpts = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
t0 = time.time()
trainer.train(resume_from_checkpoint=ckpts[-1] if ckpts else None)
print("Elapsed (min):", round((time.time() - t0) / 60, 1))
print("Peak GPU mem (GB):", round(torch.cuda.max_memory_allocated() / 1e9, 2))

Step,Training Loss,Validation Loss


Elapsed (min): 0.0
Peak GPU mem (GB): 1.31


In [23]:
# 5.4 save adapter to Drive
trainer.model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
print("Saved to", ADAPTER_DIR)

Saved to /content/drive/MyDrive/Colab Notebooks/Qareed/Qareed-Qwen2.5-1.5B-Poet-v1/adapter


## 6) Evaluation: before vs after

Read the outputs yourself. Check: (1) number of baits, (2) does the theme match, (3) rhyme, (4) no foreign text / explanations.
The automatic metrics below are rough (rhyme = last letter only; **meter is not measured**).

In [24]:
# 6.1 side-by-side: base (adapter disabled) vs tuned, same prompts
trainer.model.config.use_cache = True
for p in TEST_PROMPTS:
    print("=" * 70); print("Q:", p)
    with trainer.model.disable_adapter():
        print("\n[BASE]\n", generate(p, trainer.model))
    print("\n[TUNED]\n", generate(p, trainer.model))

Q: اكتب بيتين من الشعر في الحزن.

[BASE]
 عذراً، كمساعد ق�دي، لا يمكنني كتابة الشعر أو الشعور بالحزن. يتعين علي الابتعاد عن الفردوس العاطفي الذي يتكون في أعمق أعماق الشخصيات البشرية. ولكن إذا كنت تريد شهادة عاطفية لغة عربية ناجحة، أنا هنا لتساعدك. الرجاء التوضيح بما تحتاج.

[TUNED]
 ومنيت على حبيب  ...  يبدي لي الحزن
أحبابي انجلي قلبي  ...  بحبه واصلني
Q: انظم ثلاثة أبيات في الفخر.

[BASE]
 في خيال الغامض وفي غروب الليل أتمنى أن نرى القمر يغيب بسلام ويأتي يوم جديد يحمل السعادة والسعادة.

[TUNED]
 وذا رجعت إلى بيت سيدكم  ...  فقد أضحى خير عارضكم قومه
ولما أتيت أبا بكر بن عتبة  ...  فاق ليها نجله ملكا يحكم
فقلت هذا ابن حرب قد ذهب  ...  إلى الله وقد جمعت فيه مسلميه
Q: اكتب بيتًا واحدًا عن الصبر.

[BASE]
 ومع الخيبة، يزداد الصبر رمادًا، ولا تُخيب العواطف إلا الصبر.

[TUNED]
 لا تنكرن أبدا صبري  ...  ولا ينقصني من حسن العطايا
Q: أريد بيتين من الشعر العربي في الغزل.

[BASE]
 قد يكون "يا غزورة يا غزوة" أو "يا غصن يا زهرة" قديماً ولكن لا أستطيع تحديد أي منها بالضبط بدون مشاهدة. لكن يمكنك كتابة

In [25]:
# 6.2 simple metrics on the validation samples
from contextlib import nullcontext

def last_letter(bait):
    words = DIAC.sub("", bait).strip().split()
    if not words: return None
    w = re.sub(r"[^\u0621-\u064A]", "", words[-1])
    return w[-1] if w else None

def rhyme_consistency(text):
    letters = [l for l in (last_letter(b) for b in text.split("\n") if b.strip()) if l]
    if len(letters) < 2: return None
    return collections.Counter(letters).most_common(1)[0][1] / len(letters)

def count_baits(text):
    return len([b for b in text.split("\n") if b.strip()])

def evaluate(use_adapter, n=60):
    ctx = nullcontext() if use_adapter else trainer.model.disable_adapter()
    rows = []
    with ctx:
        for s in val_samples[:n]:
            out = generate(s["prompt"], trainer.model, temperature=0.7)
            rows.append({"count_ok": count_baits(out) == s["n_baits"],
                         "no_latin": not LATIN_RE.search(out),
                         "rhyme": rhyme_consistency(out)})
    d = pd.DataFrame(rows)
    return {"count_match": round(d.count_ok.mean(), 3), "no_latin": round(d.no_latin.mean(), 3),
            "rhyme": round(d.rhyme.dropna().mean(), 3)}

print("BASE :", evaluate(False))
print("TUNED:", evaluate(True))

# reference ceiling: rhyme consistency of the REAL poetry answers
ref = [rhyme_consistency(s["answer"]) for s in val_samples[:200] if s["n_baits"] >= 2]
print("REAL DATA rhyme (reference):", round(float(np.mean([r for r in ref if r is not None])), 3))

BASE : {'count_match': np.float64(0.317), 'no_latin': np.float64(0.983), 'rhyme': np.float64(0.39)}
TUNED: {'count_match': np.float64(1.0), 'no_latin': np.float64(1.0), 'rhyme': np.float64(0.653)}
REAL DATA rhyme (reference): 0.905


## 7) Publish to Hugging Face (private by default)

Review section 6 first. Then run 7.1 (adapter), then 7.2 (merged model). 7.2 merges on **CPU** to avoid GPU/RAM crashes.

In [26]:
# 7.1 push adapter
trainer.model.push_to_hub(ADAPTER_REPO, private=PUSH_PRIVATE)
tok.push_to_hub(ADAPTER_REPO, private=PUSH_PRIVATE)
print("https://huggingface.co/" + ADAPTER_REPO)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  22%|##1       | 16.0MB / 73.9MB            

Validating                    :           |   0%            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpmp4typye/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


https://huggingface.co/Youssefx64/Qareed-Qwen2.5-1.5B-Poet-v1-LoRA


In [27]:
# 7.2 merge the adapter into an fp16 base (on CPU) and push the full model
# Upgrade torchao first to resolve the PEFT incompatibility error
!pip install -q -U torchao

import gc
import torch
from transformers import AutoModelForCausalLM

for name in ["trainer", "model"]:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()

from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map="cpu")
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()

merged.push_to_hub(MERGED_REPO, private=PUSH_PRIVATE)
tok.push_to_hub(MERGED_REPO, private=PUSH_PRIVATE)
print("https://huggingface.co/" + MERGED_REPO)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpo7ft03cv/tokenizer.json:  70%|######9   | 7.99MB / 11.4MB            

Validating                    :           |   0%            

https://huggingface.co/Youssefx64/Qareed-Qwen2.5-1.5B-Poet-v1


In [28]:
# 7.3 model card (state the limitations honestly)
from huggingface_hub import ModelCard
F = "`" * 3     # avoids nesting code fences inside this cell

card_text = f"""---
language:
- ar
license: apache-2.0
base_model: {BASE_MODEL}
datasets:
- arbml/Ashaar_dataset
library_name: transformers
pipeline_tag: text-generation
tags:
- arabic
- poetry
- qlora
- peft
---

# Qareed Poet 1.5B (v1) — قريض

Arabic poetry generation model, QLoRA fine-tuned from `{BASE_MODEL}`. This is an early experimental release (v1).

## Usage
{F}python
from transformers import pipeline
pipe = pipeline("text-generation", model="{MERGED_REPO}", device_map="auto")
msgs = [{{"role": "user", "content": "اكتب بيتين من الشعر في الشوق."}}]
print(pipe(msgs, max_new_tokens=120, do_sample=True, temperature=0.8)[0]["generated_text"][-1]["content"])
{F}

## Training
- Data: `arbml/Ashaar_dataset` (Ashaar, arXiv:2307.06218): cleaned, deduplicated, poem-level train/val split, colloquial poems removed
- Method: QLoRA (NF4), r=16, alpha=32, all linear layers, 1 epoch, ~20k instruction samples (1-4 consecutive baits)
- Diacritics were stripped; meter was NOT used as a control signal in v1
- Final validation loss: about 1.337

## What it does well
- Follows the requested number of baits and the sadr/ajz format; answers without explanations.

## Known limitations
- Poetic quality is limited: rhyme is often inconsistent, meter (wazn) is not controlled or checked, and the requested theme is frequently not respected.
- Source text lost the letter alef-with-hamza-below; common words were patched, but some words may still be wrong.
- Theme labels come from the source websites and are noisy.
- May reproduce lines from the training data. Outputs need human review.
"""
ModelCard(card_text).push_to_hub(MERGED_REPO)
print("Model card pushed.")

Model card pushed.


## 8) Quick inference test of the merged model (optional)

In [14]:
# 8.1 load the merged model from the Hub and try it
from transformers import pipeline
pipe = pipeline("text-generation", model="MERGED_REPO", device_map="auto", dtype=torch.float16)
msgs = [{"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "اكتب بيتين من الشعر في الشوق والحنين."}]
print(pipe(msgs, max_new_tokens=120, do_sample=True, temperature=0.8)[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


يا ملك الدهر إن أهوى جسمي  ...  لست فيه حي ولا هو في حميتي
لا شك أنني أحبك بعشق  ...  لم يزل يزداد وأنت تزيد


In [5]:
from transformers import pipeline
pipe = pipeline("text-generation", model="Youssefx64/Qareed-Qwen2.5-1.5B-Poet-v1", device_map="auto")
msgs = [{"role": "user", "content": "اكتب بيتين من الشعر في الشوق."}]
print(pipe(msgs, max_new_tokens=120, do_sample=True, temperature=0.8)[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


لولا الشوق ما هوى من حسنكم  ...  وأحببت لكم من نظرة عيوني
ما ضرني لو أن قلبي لم يهوى  ...  إلى أحد غيركم فكم أنت طيب
